# CNN Paper-Style：先做健康检查，再换一个有信号的目标

上一轮诊断的结论是，**下一根 5 分钟 K 线的涨跌几乎没有可学的信号**（梯度提升树加手工特征在验证集上的 AUC 只有约 0.507）。所以原来的 CNN78u 学成了“永远猜涨”。这个 notebook 用同一份 `cnn78u_colab_bundle.zip`，不需要 WRDS，做两件事：

| 实验 | 标签 | 作用 |
|---|---|---|
| `vol_h1` | 下一根的 \|收益\| 是否大于该股**训练期**中位数 | **管道健康检查**：这个目标有明确信号，手工特征的 AUC 约 0.67。CNN 应该能学到 0.6 以上；学不到就说明网络或训练本身有问题。 |
| `xsdir_h12` | 未来 12 根（1 小时）的收益是否**跑赢同一时刻所有股票的中位数** | 真正要研究的方向问题。横截面标签去掉了市场整体的涨跌，类别天然接近 50/50。 |

**和原来 CNN78u 的区别，按 Jiang–Kelly–Xiu 原文对齐：**
- 架构方面，沿用原文的模块：卷积核 5×3，接 BN 和 LeakyReLU，**只在纵向做 2×1 max-pool**，第一层 dilation (2,1)，最后 flatten 后经过 dropout 0.5 接全连接层。针对 234×234、78 根 K 线的图做了两处适配：
  - 第一层横向 stride 设为 3。每根 K 线正好占 3 个像素宽，所以输出的每一列对应一根 K 线，**78 根的时间分辨率一直保留到分类器**，不再像原来那样用 4×4 池化把最后几根 K 线平均掉。
  - 进入全连接层之前，只在纵向做平均池化，压到 4 行。这样全连接层的输入约 8 万维，和原文 20 天模型（约 4.6 万维）同一量级。如果直接 flatten，会有 60 万维；在 Adam 下，每一步 logit 都会被推得太远，我在本地测试时就看到 loss 涨到了 1.0 以上。
- 初始化方面，卷积和全连接层用 Xavier，**输出层权重置零**，所以初始 loss 正好是 ln2 ≈ 0.693，不会再出现第一轮 0.84 的情况。
- 优化方面，Adam，**lr = 1e-5**，batch 128，weight decay 0。按验证集 log loss 做 early stopping，patience 2。
- 样本方面，stride = 6（每 30 分钟取一个决策点），减少相邻窗口 77/78 重叠带来的冗余，每个 epoch 也快很多。
- 基线方面，对比训练集先验（log loss 基准）、只看输入的简单规则，以及**用手工特征训练的梯度提升树**。CNN 只有超过这棵树，才说明图片里有额外的信息。
- 评估方面，看 AUC、相对先验的 log loss 改善，以及**同一时刻按预测概率分十组的多空收益**（top 减 bottom，单位 bps，按日求 t 值）。

**怎么跑：** 把本 notebook 放到 `MyDrive/Colab Notebooks/EC/Project/Demo`，和 `cnn78u_colab_bundle.zip` 放在一起。选 **GPU 运行时**（L4 即可），然后依次运行所有 cell。默认不评估测试集。

## 1. 依赖

In [ ]:
import os, sys, subprocess, importlib.util
from pathlib import Path
LOCAL_TEST = os.environ.get("CNN78U_LOCAL_NOTEBOOK_TEST") == "1"
packages = {"numpy":"numpy", "pandas":"pandas", "pyarrow":"pyarrow", "sklearn":"scikit-learn",
            "matplotlib":"matplotlib"}
missing = [pkg for mod, pkg in packages.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
if importlib.util.find_spec("torch") is None:
    raise RuntimeError("PyTorch is missing. Select a standard Colab GPU runtime.")

## 2. 挂载 Drive

In [ ]:
if LOCAL_TEST:
    DRIVE_DEMO = Path(os.environ["CNN78U_LOCAL_DEMO"])
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DEMO = Path("/content/drive/MyDrive/Colab Notebooks/EC/Project/Demo")
assert DRIVE_DEMO.is_dir(), f"Folder not found: {DRIVE_DEMO}"
print("Drive project:", DRIVE_DEMO)

## 3. 把 bundle 解压到运行时本地磁盘
训练直接读本地 memmap，避免对 Drive 做大量小文件读写。

In [ ]:
import hashlib, json, shutil, tempfile, time, zipfile
LOCAL_CACHE = Path(tempfile.mkdtemp(prefix="paper_", dir=None if LOCAL_TEST else "/content"))
LOCAL_PROJECT = LOCAL_CACHE / "project"; LOCAL_PROJECT.mkdir()
bundle = next((p for p in [DRIVE_DEMO / "cnn78u_colab_bundle.zip",
                           DRIVE_DEMO / "colab/cnn78u_colab_bundle.zip"] if p.is_file()), None)
assert bundle is not None, "Put cnn78u_colab_bundle.zip next to this notebook"
started = time.perf_counter()
shutil.copy2(bundle, LOCAL_CACHE / "bundle.zip")
with zipfile.ZipFile(LOCAL_CACHE / "bundle.zip") as z:
    for info in z.infolist():
        assert (LOCAL_PROJECT / info.filename).resolve().is_relative_to(LOCAL_PROJECT.resolve())
    z.extractall(LOCAL_PROJECT)
(LOCAL_CACHE / "bundle.zip").unlink()
manifest_path = LOCAL_PROJECT / "bundle_manifest.json"
if manifest_path.exists():
    for rel, d in json.loads(manifest_path.read_text())["files"].items():
        assert hashlib.sha256((LOCAL_PROJECT / rel).read_bytes()).hexdigest() == d["sha256"], rel
sys.path.insert(0, str(LOCAL_PROJECT))
PREPARED = LOCAL_PROJECT / "prepared"
print({"prepared": str(PREPARED), "staging_seconds": round(time.perf_counter() - started, 1)})

## 4. 写入实验代码
下面的模块只依赖 bundle 里的 `us_pipeline/images.py`（与原来完全相同的作图函数），其余逻辑都在这里，方便你直接阅读和修改。

In [ ]:
RUNNER_SOURCE = "\"\"\"Paper-style chart CNN on the existing 5-minute bundle: new targets, honest baselines.\n\nTargets are rebuilt from the prepared bundle (no WRDS access needed):\n  vol    : 1 if |r(t -> t+H)| exceeds that stock's TRAINING median |r| at the same horizon\n  xs_dir : 1 if r(t -> t+H) beats the cross-sectional median return at the same decision time\n  dir    : 1 if r(t -> t+H) > 0 (raw direction; zero returns dropped)\nInputs are the same 78-bar OHLCV chart images as CNN78u (window-local price axis).\n\"\"\"\nimport json\nimport os\nimport random\nimport shutil\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader, Dataset\nfrom sklearn.ensemble import HistGradientBoostingClassifier\nfrom sklearn.metrics import accuracy_score, balanced_accuracy_score, log_loss, roc_auc_score\n\nfrom us_pipeline.images import prediction_basis, render_window\n\nLOOKBACK = 78\n\n\n# ----------------------------------------------------------------------------- data\nclass Store:\n    \"\"\"Memory-light access to the prepared bundle.\"\"\"\n\n    def __init__(self, prepared):\n        self.root = Path(prepared)\n        self.manifest = json.loads((self.root / \"manifest.json\").read_text())\n        self.config = self.manifest[\"config\"]\n        assert self.config[\"lookback\"] == LOOKBACK and self.config[\"bar_minutes\"] == 5\n        features = np.load(self.root / \"features.npy\", mmap_mode=\"r\")\n        self.close = np.array(features[:, :, 3])          # ~40 MB\n        self.volume = np.array(features[:, :, 4])\n        self.log_close = np.log(self.close)\n        valid = np.load(self.root / \"valid.npy\")\n        self.bad_cumsum = np.concatenate(\n            [np.zeros((valid.shape[0], 1), np.int32), np.cumsum(~valid, axis=1, dtype=np.int32)], axis=1)\n        clock = pd.to_datetime(self.manifest[\"decision_clock_ns\"], unit=\"ns\", utc=True)\n        self.dates = np.array(clock.tz_convert(\"America/New_York\").strftime(\"%Y-%m-%d\"))\n        self.T = self.close.shape[1]\n\n    def split_frame(self, split):\n        return pd.read_parquet(self.root / f\"{split}.parquet\",\n                               columns=[\"security_index\", \"permno\", \"end_index\", \"decision_time\"])\n\n\ndef build_split(store, split, target, horizon, stride, train_abs_median=None):\n    \"\"\"Return a sample frame with labels for one split. Inputs reuse the audited windows.\"\"\"\n    f = store.split_frame(split)\n    f = f[f.end_index % stride == stride - 1].reset_index(drop=True)\n    s = f.security_index.to_numpy(np.int64)\n    t = f.end_index.to_numpy(np.int64)\n    h = int(horizon)\n    inside = t + h < store.T\n    s, t, f = s[inside], t[inside], f[inside].reset_index(drop=True)\n    # every future bar t+1..t+H must be observed\n    future_ok = (store.bad_cumsum[s, t + h + 1] - store.bad_cumsum[s, t + 1]) == 0\n    # purge: label end must stay inside the same chronological split\n    end_dates = store.dates[t + h]\n    cfg = store.config\n    if split == \"train\":\n        purge_ok = end_dates <= cfg[\"train_end\"]\n    elif split == \"validation\":\n        purge_ok = end_dates <= cfg[\"validation_end\"]\n    else:\n        purge_ok = np.ones(len(t), bool)\n    keep = future_ok & purge_ok\n    f, s, t = f[keep].reset_index(drop=True), s[keep], t[keep]\n    r = store.close[s, t + h] / store.close[s, t] - 1.0\n    f[\"forward_return\"] = r\n    f[\"xs_median\"] = f.groupby(\"end_index\").forward_return.transform(\"median\")\n    f[\"xs_return\"] = r - f.xs_median.to_numpy()\n    f[\"decision_date\"] = store.dates[t]\n    if target == \"dir\":\n        f = f[f.forward_return != 0]\n        f[\"label\"] = (f.forward_return > 0).astype(\"int64\")\n    elif target == \"xs_dir\":\n        counts = f.groupby(\"end_index\").end_index.transform(\"size\")\n        f = f[(counts >= 20) & (f.xs_return != 0)]\n        f[\"label\"] = (f.xs_return > 0).astype(\"int64\")\n    elif target == \"vol\":\n        absr = f.forward_return.abs()\n        if train_abs_median is None:\n            train_abs_median = absr.groupby(f.security_index).median()\n        med = f.security_index.map(train_abs_median)\n        f = f[med.notna() & (absr != med)]\n        f[\"label\"] = (f.forward_return.abs() > f.security_index.map(train_abs_median)).astype(\"int64\")\n    else:\n        raise ValueError(\"target must be vol, xs_dir or dir\")\n    return f.reset_index(drop=True), train_abs_median\n\n\nclass ChartDataset(Dataset):\n    def __init__(self, prepared, frame, include_volume=True):\n        self.root = Path(prepared)\n        self.s = frame.security_index.to_numpy(np.int64)\n        self.t = frame.end_index.to_numpy(np.int64)\n        self.y = frame.label.to_numpy(np.int64)\n        self.include_volume = include_volume\n        self._x = self._m = None\n\n    def __len__(self):\n        return len(self.y)\n\n    def __getstate__(self):\n        state = self.__dict__.copy()\n        state[\"_x\"] = state[\"_m\"] = None\n        return state\n\n    def image(self, i):\n        if self._x is None:\n            self._x = np.load(self.root / \"features.npy\", mmap_mode=\"r\")\n            self._m = np.load(self.root / \"multipliers.npy\", mmap_mode=\"r\")\n        s, t = self.s[i], self.t[i]\n        window = prediction_basis(self._x[s, t - LOOKBACK + 1:t + 1], self._m[s, t])\n        return render_window(window, self.include_volume)\n\n    def __getitem__(self, i):\n        x = torch.from_numpy(self.image(i).astype(np.float32)[None] / 255.0)\n        return x, int(self.y[i]), int(i)\n\n\n# ----------------------------------------------------------------------------- model\nclass PaperCNN(nn.Module):\n    \"\"\"Jiang-Kelly-Xiu style blocks: 5x3 conv, BN, LeakyReLU, vertical-only 2x1 max-pool;\n    first layer dilation (2,1). Two deliberate adaptations to the 234x234 / 78-bar image:\n      * first-layer stride (3,3): each bar is exactly 3 pixels wide, so every output column\n        corresponds to one bar (78 columns kept all the way to the classifier);\n      * height-only average pool to `head_height` rows before the linear head, keeping the\n        flattened fan-in (~80k) close to the paper's 20-day model (~46k) instead of ~600k.\n        With Adam, a huge fan-in makes each step move the logit too far and training noisy.\"\"\"\n\n    def __init__(self, channels=(64, 128, 256), dropout=0.5, head_height=4, input_hw=(234, 234)):\n        super().__init__()\n        layers, c_in = [], 1\n        for i, c in enumerate(channels):\n            if i == 0:\n                conv = nn.Conv2d(c_in, c, (5, 3), stride=(3, 3), dilation=(2, 1), padding=(12, 0))\n            else:\n                conv = nn.Conv2d(c_in, c, (5, 3), padding=(2, 1))\n            layers += [conv, nn.BatchNorm2d(c), nn.LeakyReLU(0.01), nn.MaxPool2d((2, 1))]\n            c_in = c\n        layers.append(nn.AdaptiveAvgPool2d((head_height, None)))\n        self.features = nn.Sequential(*layers)\n        with torch.no_grad():\n            shape = self.features(torch.zeros(1, 1, *input_hw)).shape\n        self.feature_shape = tuple(shape[1:])\n        self.flat_features = int(np.prod(self.feature_shape))\n        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(self.flat_features, 2))\n        for m in self.modules():\n            if isinstance(m, (nn.Conv2d, nn.Linear)):\n                nn.init.xavier_uniform_(m.weight)\n                nn.init.zeros_(m.bias)\n        # Zero output layer: initial prediction is exactly 50/50, initial loss = ln 2.\n        nn.init.zeros_(self.head[-1].weight)\n\n    def forward(self, x):\n        return self.head(self.features(x))\n\n\n# ----------------------------------------------------------------------------- helpers\ndef seed_all(seed):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef _seed_worker(_):\n    seed = torch.initial_seed() % 2**32\n    np.random.seed(seed); random.seed(seed)\n\n\ndef make_loader(ds, batch_size, workers, shuffle, seed):\n    kw = dict(batch_size=batch_size, shuffle=shuffle, num_workers=workers,\n              pin_memory=torch.cuda.is_available(), worker_init_fn=_seed_worker,\n              generator=torch.Generator().manual_seed(seed))\n    if workers:\n        kw.update(persistent_workers=True, prefetch_factor=4)\n    return DataLoader(ds, **kw)\n\n\ndef mirror(path, persistent):\n    if persistent is None:\n        return\n    persistent = Path(persistent); persistent.mkdir(parents=True, exist_ok=True)\n    tmp = persistent / (path.name + \".copying\")\n    shutil.copy2(path, tmp); os.replace(tmp, persistent / path.name)\n\n\ndef save_json(path, value, persistent=None):\n    path.write_text(json.dumps(value, indent=2, default=float) + \"\\n\"); mirror(path, persistent)\n\n\n@torch.no_grad()\ndef predict(model, loader, device, progress=None):\n    model.eval(); out = np.empty(len(loader.dataset), np.float64)\n    for x, _, idx in loader:\n        p = model(x.to(device, non_blocking=True)).float().softmax(1)[:, 1].cpu().numpy()\n        out[idx.numpy()] = p\n    return out\n\n\ndef metrics(y, p, prior=None):\n    p = np.clip(np.asarray(p, float), 1e-7, 1 - 1e-7); y = np.asarray(y)\n    m = dict(samples=len(y), positive_fraction=float(y.mean()),\n             accuracy=float(accuracy_score(y, p >= .5)),\n             balanced_accuracy=float(balanced_accuracy_score(y, p >= .5)),\n             roc_auc=float(roc_auc_score(y, p)), log_loss=float(log_loss(y, p, labels=[0, 1])),\n             prob_std=float(p.std()))\n    if prior is not None:\n        base = log_loss(y, np.full(len(y), prior), labels=[0, 1])\n        m[\"log_loss_vs_prior_pct\"] = float(100 * (base - m[\"log_loss\"]) / base)\n    return m\n\n\n# ----------------------------------------------------------------------------- baselines\ndef hand_features(store, frame):\n    s = frame.security_index.to_numpy(np.int64); t = frame.end_index.to_numpy(np.int64)\n    lc = store.log_close\n    ret = lambda k: lc[s, t - k + 1] - lc[s, t - k]\n    cols = {f\"r{k}\": ret(k) for k in range(1, 13)}\n    for w in (6, 12, 39, 77):\n        cols[f\"cum{w}\"] = lc[s, t] - lc[s, t - w]\n    past = np.stack([ret(k) for k in range(1, 78)])\n    cols[\"rv77\"] = np.nanstd(past, axis=0)\n    cols[\"absr12\"] = np.nanmean(np.abs(past[:12]), axis=0)\n    vol = store.volume\n    cols[\"vrel\"] = np.log(vol[s, t] / np.nanmean(np.stack([vol[s, t - k] for k in range(1, 78)]), axis=0))\n    cols[\"tod\"] = t % 78\n    return pd.DataFrame(cols)\n\n\ndef baselines(store, target, train, valid, seed, max_train=300_000):\n    \"\"\"Return {name: validation score array}; rules use inputs only, HGB is fit on training only.\"\"\"\n    out = {}\n    prior = float(train.label.mean())\n    out[\"training_prior\"] = np.full(len(valid), prior)\n    Fva = hand_features(store, valid)\n    if target == \"vol\":\n        out[\"rule_recent_abs_return\"] = Fva.absr12.to_numpy()\n    else:\n        out[\"rule_last_bar_momentum\"] = Fva.r1.to_numpy()\n        out[\"rule_78bar_momentum\"] = Fva.cum77.to_numpy()\n    sub = train.sample(min(max_train, len(train)), random_state=seed)\n    hgb = HistGradientBoostingClassifier(max_iter=300, learning_rate=.05, min_samples_leaf=500,\n                                         random_state=seed)\n    hgb.fit(hand_features(store, sub), sub.label)\n    out[\"hgb_hand_features\"] = hgb.predict_proba(Fva)[:, 1]\n    return out, prior\n\n\ndef score_table(y, scores, prior):\n    rows = []\n    for name, p in scores.items():\n        p = np.nan_to_num(np.asarray(p, float), nan=np.nanmedian(p))\n        probability = name in (\"training_prior\", \"hgb_hand_features\", \"CNN\")\n        if probability:\n            rows.append(dict(name=name, **metrics(y, p, prior)))\n        else:  # rule score: only ranking is meaningful\n            rows.append(dict(name=name, samples=len(y), roc_auc=float(roc_auc_score(y, p))))\n    return pd.DataFrame(rows)\n\n\ndef decile_report(frame, p, target):\n    \"\"\"Within each decision time, sort stocks by predicted probability into deciles.\"\"\"\n    f = frame[[\"end_index\", \"decision_date\", \"forward_return\", \"xs_return\"]].copy()\n    f[\"p\"] = p\n    f[\"decile\"] = f.groupby(\"end_index\").p.transform(\n        lambda x: pd.qcut(x.rank(method=\"first\"), 10, labels=False) if len(x) >= 20 else np.nan)\n    f = f.dropna(subset=[\"decile\"])\n    value = f.forward_return.abs() if target == \"vol\" else f.xs_return\n    f[\"value_bps\"] = value * 1e4\n    table = f.groupby(\"decile\").value_bps.agg([\"mean\", \"count\"]).reset_index()\n    spread = None\n    if target != \"vol\":\n        g = f.groupby([\"decision_date\", \"end_index\", \"decile\"]).value_bps.mean().unstack()\n        ls = (g[9] - g[0]).groupby(level=0).mean()            # one number per day\n        spread = dict(days=int(len(ls)), mean_bps=float(ls.mean()),\n                      t_stat_daily=float(ls.mean() / (ls.std(ddof=1) / np.sqrt(len(ls)))),\n                      note=\"Top-minus-bottom decile, gross of costs; per-day averages\")\n    return table, spread\n\n\n# ----------------------------------------------------------------------------- training\ndef run_experiment(prepared, exp, settings, local_dir, persistent_dir, device, smoke=False):\n    local_dir = Path(local_dir); local_dir.mkdir(parents=True, exist_ok=True)\n    seed = settings[\"seed\"]; seed_all(seed)\n    store = Store(prepared)\n    stride = settings[\"stride\"]\n    train, med = build_split(store, \"train\", exp[\"target\"], exp[\"horizon\"], stride)\n    valid, _ = build_split(store, \"validation\", exp[\"target\"], exp[\"horizon\"], stride, med)\n    if smoke:\n        train = train.sample(min(256, len(train)), random_state=seed).reset_index(drop=True)\n        valid = valid.groupby(\"end_index\").filter(lambda g: len(g) >= 20)\n        valid = valid[valid.end_index.isin(valid.end_index.drop_duplicates().sample(2, random_state=seed))]\n        valid = valid.reset_index(drop=True)\n    info = dict(experiment=exp, settings=settings, train_samples=len(train), validation_samples=len(valid),\n                train_positive=float(train.label.mean()), validation_positive=float(valid.label.mean()))\n    print(json.dumps(info, default=str))\n    save_json(local_dir / \"experiment.json\", info, persistent_dir)\n    if med is not None:\n        med.to_csv(local_dir / \"train_abs_return_median.csv\"); mirror(local_dir / \"train_abs_return_median.csv\", persistent_dir)\n\n    print(\"Fitting baselines (training data only)...\")\n    base_scores, prior = baselines(store, exp[\"target\"], train, valid, seed,\n                                   max_train=2000 if smoke else 300_000)\n\n    model = PaperCNN(channels=tuple(settings[\"channels\"]), dropout=settings[\"dropout\"],\n                     head_height=settings[\"head_height\"]).to(device)\n    print(\"Parameters:\", sum(p.numel() for p in model.parameters()), \"| feature map:\", model.feature_shape)\n    opt = torch.optim.Adam(model.parameters(), lr=settings[\"learning_rate\"], weight_decay=settings[\"weight_decay\"])\n    tr_ds, va_ds = ChartDataset(prepared, train), ChartDataset(prepared, valid)\n    tr_loader = make_loader(tr_ds, settings[\"batch_size\"], settings[\"num_workers\"], True, seed)\n    va_loader = make_loader(va_ds, settings[\"eval_batch_size\"], settings[\"num_workers\"], False, seed)\n\n    history, best, bad, start_epoch = [], float(\"inf\"), 0, 1\n    if (local_dir / \"last.pt\").exists():\n        ck = torch.load(local_dir / \"last.pt\", map_location=\"cpu\", weights_only=False)\n        model.load_state_dict(ck[\"model\"]); opt.load_state_dict(ck[\"optimizer\"])\n        history, best, bad, start_epoch = ck[\"history\"], ck[\"best\"], ck[\"bad\"], ck[\"epoch\"] + 1\n        print(f\"Resumed after epoch {ck['epoch']}\")\n        if ck.get(\"finished\"):\n            start_epoch = settings[\"epochs\"] + 1\n\n    log_every = max(1, len(tr_loader) // 10)\n    for epoch in range(start_epoch, settings[\"epochs\"] + 1):\n        model.train(); tr_loader.generator.manual_seed(seed + epoch)\n        t0 = time.perf_counter(); seen = loss_sum = correct = 0\n        for b, (x, y, _) in enumerate(tr_loader):\n            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)\n            logits = model(x); loss = nn.functional.cross_entropy(logits, y)\n            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()\n            seen += len(y); loss_sum += loss.item() * len(y); correct += int((logits.argmax(1) == y).sum())\n            if (b + 1) % log_every == 0:\n                print(f\"  epoch {epoch} batch {b+1}/{len(tr_loader)} train_loss={loss_sum/seen:.5f} \"\n                      f\"acc={correct/seen:.4f} {seen/(time.perf_counter()-t0):.0f} img/s\", flush=True)\n        train_seconds = time.perf_counter() - t0\n        p = predict(model, va_loader, device)\n        m = metrics(valid.label.to_numpy(), p, prior)\n        row = dict(epoch=epoch, train_loss=loss_sum / seen, train_accuracy=correct / seen,\n                   train_seconds=train_seconds, **{\"val_\" + k: v for k, v in m.items()})\n        history.append(row); print(json.dumps(row))\n        improved = m[\"log_loss\"] < best - settings[\"min_delta\"]\n        if m[\"log_loss\"] < best:\n            best = m[\"log_loss\"]\n            torch.save(dict(model=model.state_dict(), epoch=epoch, settings=settings, experiment=exp),\n                       local_dir / \"best.pt\"); mirror(local_dir / \"best.pt\", persistent_dir)\n            np.save(local_dir / \"best_validation_probability.npy\", p)\n        bad = 0 if improved else bad + 1\n        finished = bad >= settings[\"patience\"] or epoch == settings[\"epochs\"]\n        torch.save(dict(model=model.state_dict(), optimizer=opt.state_dict(), history=history, best=best,\n                        bad=bad, epoch=epoch, finished=finished), local_dir / \"last.pt\")\n        mirror(local_dir / \"last.pt\", persistent_dir)\n        pd.DataFrame(history).to_csv(local_dir / \"history.csv\", index=False)\n        mirror(local_dir / \"history.csv\", persistent_dir)\n        if bad >= settings[\"patience\"]:\n            print(f\"Early stop: no validation log-loss improvement for {bad} epochs\"); break\n\n    p_best = np.load(local_dir / \"best_validation_probability.npy\")\n    scores = {\"CNN\": p_best, **base_scores}\n    table = score_table(valid.label.to_numpy(), scores, prior)\n    table.to_csv(local_dir / \"validation_comparison.csv\", index=False)\n    deciles, spread = decile_report(valid, p_best, exp[\"target\"])\n    deciles.to_csv(local_dir / \"validation_deciles.csv\", index=False)\n    hgb_deciles, hgb_spread = decile_report(valid, base_scores[\"hgb_hand_features\"], exp[\"target\"])\n    summary = dict(experiment=exp, best_epoch=int(pd.DataFrame(history).sort_values(\"val_log_loss\").epoch.iloc[0]),\n                   epochs_run=len(history), cnn_long_short=spread, hgb_long_short=hgb_spread,\n                   validation=table.to_dict(\"records\"), smoke_only=smoke, test_evaluated=False)\n    save_json(local_dir / \"summary.json\", summary, persistent_dir)\n    for name in [\"validation_comparison.csv\", \"validation_deciles.csv\"]:\n        mirror(local_dir / name, persistent_dir)\n    return dict(summary=summary, history=pd.DataFrame(history), table=table, deciles=deciles,\n                store=store, median=med, prior=prior)\n\n\ndef evaluate_test(prepared, exp, settings, local_dir, persistent_dir, device, median, prior, smoke=False):\n    \"\"\"One-shot held-out evaluation of best.pt. Do not tune anything after looking at it.\"\"\"\n    store = Store(prepared)\n    test, _ = build_split(store, \"test\", exp[\"target\"], exp[\"horizon\"], settings[\"stride\"], median)\n    if smoke:\n        test = test[test.end_index.isin(test.end_index.drop_duplicates().sample(2, random_state=0))].reset_index(drop=True)\n    model = PaperCNN(channels=tuple(settings[\"channels\"]), dropout=settings[\"dropout\"],\n                     head_height=settings[\"head_height\"]).to(device)\n    model.load_state_dict(torch.load(Path(local_dir) / \"best.pt\", map_location=device, weights_only=False)[\"model\"])\n    loader = make_loader(ChartDataset(prepared, test), settings[\"eval_batch_size\"], settings[\"num_workers\"], False, 0)\n    p = predict(model, loader, device)\n    m = metrics(test.label.to_numpy(), p, prior)\n    deciles, spread = decile_report(test, p, exp[\"target\"])\n    result = dict(experiment=exp, test=m, long_short=spread, smoke_only=smoke)\n    save_json(Path(local_dir) / \"test_summary.json\", result, persistent_dir)\n    return result, deciles\n"
(LOCAL_PROJECT / "paper_runner.py").write_text(RUNNER_SOURCE)
import importlib; importlib.invalidate_caches()
import paper_runner as pr
importlib.reload(pr)
print("paper_runner loaded")

## 5. 设置
要改超参数就改这里。`RESUME=True` 并填回原来的 `RUN_ID`，可以从最近一个完整 epoch 接着跑（未完成的 epoch 会重跑）。

In [ ]:
import numpy as np, pandas as pd, torch
from datetime import datetime, timezone
from IPython.display import display
SMOKE_MODE = LOCAL_TEST            # True = tiny code check only, never a result
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda" and not SMOKE_MODE:
    raise RuntimeError("Select a GPU runtime (Runtime > Change runtime type > GPU)")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True

SETTINGS = dict(
    learning_rate = 1e-5,          # paper value; try 3e-5 / 1e-4 only as a separate, predeclared run
    weight_decay  = 0.0,
    batch_size    = 4 if SMOKE_MODE else 128,
    eval_batch_size = 8 if SMOKE_MODE else 512,
    epochs        = 1 if SMOKE_MODE else 12,
    patience      = 2,             # early stop after 2 epochs without val log-loss improvement
    min_delta     = 1e-5,
    stride        = 6,             # one decision point every 30 minutes per stock
    channels      = [64, 128, 256],
    head_height   = 4,             # rows kept by the height-only average pool before the linear head
    dropout       = 0.5,
    num_workers   = 0 if SMOKE_MODE else min(8, os.cpu_count() or 2),
    seed          = 69,
)
EXPERIMENTS = [
    dict(name="vol_h1",    target="vol",    horizon=1),    # sanity check: should clearly beat 0.5 AUC
    dict(name="xsdir_h12", target="xs_dir", horizon=12),   # real question: 1-hour cross-sectional direction
    # dict(name="dir_h12", target="dir", horizon=12),      # optional: raw direction, same horizon
]
RESUME = False
RUN_ID = "local_smoke" if LOCAL_TEST else "paper_cnn_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_utc")
PERSISTENT = DRIVE_DEMO / "paper_runs" / RUN_ID
LOCAL_RUN = LOCAL_CACHE / "run"
if RESUME:
    assert PERSISTENT.exists(), "Unknown RUN_ID"
    shutil.copytree(PERSISTENT, LOCAL_RUN)
else:
    if PERSISTENT.exists() and not LOCAL_TEST:
        raise FileExistsError("RUN_ID exists; choose a new one or set RESUME=True")
    PERSISTENT.mkdir(parents=True, exist_ok=True); LOCAL_RUN.mkdir()
pr.save_json(LOCAL_RUN / "settings.json", dict(settings=SETTINGS, experiments=EXPERIMENTS, run_id=RUN_ID,
             device=str(DEVICE), gpu=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
             torch=torch.__version__), PERSISTENT)
print("Results ->", PERSISTENT)
display(pd.Series(SETTINGS, name="settings"))

## 6. 检查模型和输入
确认初始预测正好是 0.5，并看一张输入图。

In [ ]:
import matplotlib.pyplot as plt
model = pr.PaperCNN(channels=tuple(SETTINGS["channels"]), dropout=SETTINGS["dropout"], head_height=SETTINGS["head_height"])
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters()), "| feature map (C,H,bars):", model.feature_shape)
store = pr.Store(PREPARED)
demo, _ = pr.build_split(store, "train", "xs_dir", 12, SETTINGS["stride"])
ds = pr.ChartDataset(PREPARED, demo.head(3))
x = torch.stack([ds[i][0] for i in range(3)])
with torch.no_grad():
    print("Initial P(class 1):", model.eval()(x).softmax(1)[:, 1].tolist())
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for i, ax in enumerate(axes):
    ax.imshow(x[i, 0], cmap="gray"); ax.set_title(f"permno {demo.permno[i]} | y={demo.label[i]}"); ax.axis("off")
plt.show(); del model, store, demo

## 7. 训练 + 验证
每个实验依次运行。每个 epoch 结束后都会把 `history.csv`、`best.pt`、`last.pt` 同步到 Drive。日志里要关注三样：`val_roc_auc`、`val_log_loss_vs_prior_pct`（大于 0 才说明比只猜先验好）、`val_prob_std`（接近 0 说明模型塌缩成常数）。

In [ ]:
RESULTS = {}
for exp in EXPERIMENTS:
    print("\n" + "=" * 30, exp["name"], "=" * 30)
    RESULTS[exp["name"]] = pr.run_experiment(PREPARED, exp, SETTINGS, LOCAL_RUN / exp["name"],
                                             PERSISTENT / exp["name"], DEVICE, smoke=SMOKE_MODE)
    r = RESULTS[exp["name"]]
    display(r["history"])
    print("Validation comparison (CNN vs baselines):")
    display(r["table"])
    print("Long-short (CNN):", r["summary"]["cnn_long_short"])
    print("Long-short (HGB hand features):", r["summary"]["hgb_long_short"])

## 8. 图表

In [ ]:
for name, r in RESULTS.items():
    h = r["history"]
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
    ax[0].plot(h.epoch, h.train_loss, "o-", label="train"); ax[0].plot(h.epoch, h.val_log_loss, "o-", label="validation")
    ax[0].axhline(np.log(2), ls=":", c="gray"); ax[0].set_title(f"{name}: loss"); ax[0].legend()
    ax[1].plot(h.epoch, h.val_roc_auc, "o-"); ax[1].axhline(.5, ls=":", c="gray")
    hgb = r["table"].set_index("name").loc["hgb_hand_features", "roc_auc"]
    ax[1].axhline(hgb, ls="--", c="tab:red", label=f"HGB baseline {hgb:.3f}"); ax[1].legend(); ax[1].set_title("validation AUC")
    d = r["deciles"]
    ax[2].bar(d.decile + 1, d["mean"]); ax[2].set_xlabel("CNN probability decile (within timestamp)")
    ax[2].set_title("mean |return| bps" if name.startswith("vol") else "mean excess return bps")
    fig.tight_layout(); fig.savefig(LOCAL_RUN / f"{name}_curves.png", dpi=120)
    pr.mirror(LOCAL_RUN / f"{name}_curves.png", PERSISTENT); plt.show()

## 9. 怎么读结果

1. **先看 `vol_h1`。** CNN 的验证集 AUC 应该明显大于 0.5，最好接近手工特征 HGB 的约 0.67，十分位柱状图应该单调上升。
   - 如果做到了，说明数据、作图和训练流程都没问题，接着看第 2 步。
   - 如果 AUC 还是约 0.5，而且 `val_prob_std` 接近 0，说明网络或优化有问题。这时可以先把 lr 改成 1e-4 单独跑一次 `vol_h1`，确认它能学，再继续。
2. **再看 `xsdir_h12`。** 合理的预期是 AUC 只有 0.50–0.52，这在这个任务上是正常的。真正的判断标准有三个：
   - `log_loss_vs_prior_pct` 是否大于 0；
   - CNN 的 AUC 是否超过 HGB 基线；
   - 多空 `mean_bps` 是否为正，并且 `t_stat_daily` 大于 2（验证集只有约 20 个交易日，t 值要谨慎解读）。
3. 不要根据单次结果反复调参后再看测试集。要试的变体（lr、stride、horizon）应该事先列好，每个用新的 `RUN_ID`。

## 10. （可选）测试集，一次性评估
默认关闭。你之前已经看过 5–6 月测试集的结果，所以它已经不算完全未见过的数据。只有在实验方案固定之后才打开，打开后不要再据此调参。

In [ ]:
RUN_TEST = False
if RUN_TEST:
    for exp in EXPERIMENTS:
        r = RESULTS[exp["name"]]
        test, deciles = pr.evaluate_test(PREPARED, exp, SETTINGS, LOCAL_RUN / exp["name"], PERSISTENT / exp["name"],
                                         DEVICE, r["median"], r["prior"], smoke=SMOKE_MODE)
        print(exp["name"]); display(pd.Series(test["test"])); print("Long-short:", test["long_short"]); display(deciles)
else:
    print("Test evaluation skipped (RUN_TEST=False).")

## 11. 打包结果到 Drive

In [ ]:
archive = Path(shutil.make_archive(str(LOCAL_CACHE / "paper_results"), "zip", root_dir=LOCAL_RUN))
shutil.copy2(archive, PERSISTENT / "paper_results.zip")
print("Saved:", PERSISTENT / "paper_results.zip", f"{archive.stat().st_size/1e6:.1f} MB")
print("RUN_ID (keep for RESUME):", RUN_ID)